### This ANALYSYS notebook 

- 

In [206]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [207]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [208]:
class TimeSeries(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def basic_time_series(self):
        sql = """
            SELECT publication_year AS 'publication_year', 
                    count(work_id) AS 'works count',
                    sum(cited_by_count)/count(work_id) AS 'mean cites',
                    sum(referenced_works_count)/count(work_id) AS 'mean references'
                FROM works
                WHERE (contains('article review book-chapter preprint letter', type) and type != 'book') = true
                GROUP BY publication_year
                ORDER BY publication_year
            """
        df = self.db.sql(sql).df()
        df = df.melt(id_vars='publication_year', var_name='frame', value_name='count')
        print(f'{df.shape = }\n{df.head()}')
        self.plot_basic_time_series(df=df)
        return
    
    def count_types(self):
        types = self.db.sql("""
                            SELECT type, count(work_id) AS count 
                                FROM works WHERE (contains('article review book-chapter preprint letter', type) and type != 'book') = true 
                                GROUP BY type 
                                ORDER BY count DESC""").df()
        print(f'{types.shape = }\n{types.head(9)}')
        return
    
    def plot_basic_time_series(self, df=None):
        g = sns.FacetGrid(df, row='frame', sharex=True, sharey=False)
        g.map(sns.lineplot, 'publication_year', 'count')

    def authorship_time_series(self):
        sql = """
                SELECT DISTINCT author_id, author_name, works_count_per_year, years 
                    FROM
                        (SELECT author_id,
                                list(publication_year) AS years, 
                            author_name,
                            count(w.work_id) AS works_count_per_year
                        FROM authorships a
                            LEFT JOIN works w
                            USING (work_id)
                            WHERE (contains('article review book-chapter preprint letter', w.type) and w.type != 'book') = true
                            ORDER BY publication_year
                            GROUP BY author_id, author_name)
                    ORDER BY works_count_per_year DESC
            """
        self.db.sql(sql).show()
        return

In [209]:
def main():

    ts = TimeSeries()
    # ts.basic_time_series()
    ts.authorship_time_series()

In [210]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

ParserException: Parser Error: syntax error at or near "GROUP"